In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Audio, display

import librosa
import librosa.display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from huggingface_hub import snapshot_download, login as hf_login
from datasets import load_dataset

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")

In [ ]:
ROOT_PATH = Path("../").resolve()
DATA_PATH = ROOT_PATH / "data"

## Dataset

In [ ]:
repo_id = "doof-ferb/fpt_fosd"
local_dir = "../data/fpt_fosd"

downloaded_path = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    local_dir=local_dir,
)

In [ ]:
DATASET_PATH = DATA_PATH / "fpt_fosd" / "data"
dataset = load_dataset("parquet", data_dir=DATASET_PATH)
dataset

In [ ]:
sample = dataset["train"][0]
sample

In [ ]:
audio_data = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]
print("Audio shape:", audio_data.shape)
print("Sampling rate:", sampling_rate)

In [ ]:
Audio(data=audio_data, rate=sampling_rate)

In [ ]:
# Create an array of time values (in seconds) for the x-axis
time_axis = np.linspace(0, len(audio_data) / sampling_rate, num=len(audio_data))
plt.figure(figsize=(10, 3))
plt.plot(time_axis, audio_data, color="blue")
plt.title("Audio Waveform")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.xlim(
    [0, time_axis[-1]]
)  # Ensure the x-axis starts at 0 and ends at the exact length
plt.tight_layout()
plt.show()

In [ ]:
# Create an array of time values (in seconds) for the x-axis
start_time = 2_500
end_time = 3_500
time_axis = np.linspace(
    0,
    len(audio_data[start_time:end_time]) / sampling_rate,
    num=len(audio_data[start_time:end_time]),
)
plt.figure(figsize=(10, 3))
plt.plot(time_axis, audio_data[start_time:end_time], color="cyan")
plt.title("Audio Waveform")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.xlim(
    [0, time_axis[-1]]
)  # Ensure the x-axis starts at 0 and ends at the exact length
plt.tight_layout()
plt.show()

In [ ]:
# 1. Compute the Mel Spectrogram
mel_spectrogram = librosa.feature.melspectrogram(
    y=audio_data,
    sr=sampling_rate,
    n_mels=128,  # Number of Mel bands to generate
    n_fft=2048,  # Length of the FFT window
    hop_length=512,  # Number of samples between successive frames
    fmax=8000,  # Maximum frequency (optional, often half the SR is good)
)

In [ ]:
mel_spectrogram

In [ ]:
log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)
print("Log-Mel shape:", log_mel_spectrogram.shape)
# ---------------------------------------------------------
# 3. (Optional) Visualize the Log-Mel Spectrogram
# ---------------------------------------------------------
plt.figure(figsize=(10, 4))
librosa.display.specshow(
    log_mel_spectrogram,
    sr=sampling_rate,
    hop_length=512,
    x_axis="time",
    y_axis="mel",
    fmax=8000,
)
plt.colorbar(format="%+2.0f dB")
plt.title("Log-Mel Spectrogram")
plt.tight_layout()
plt.show()

In [ ]:
# ── 1. Load audio ──────────────────────────────────────────────
# y, sr = librosa.load("audio.wav", sr=16000)  # resample to 16kHz
y = audio_data
sr = sampling_rate
# y: float32 array, normalized to [-1.0, 1.0]

# ── 2. Pre-emphasis ────────────────────────────────────────────
y_emp = np.append(y[0], y[1:] - 0.97 * y[:-1])

# ── 3. STFT → Power Spectrogram ────────────────────────────────
n_fft = 512  # ~32ms window at 16kHz
hop_len = 128  # ~8ms hop → 75% overlap
window = "hann"

D = librosa.stft(y_emp, n_fft=n_fft, hop_length=hop_len, window=window)
S_power = np.abs(D) ** 2  # shape: (257, T)

# ── 4. Mel Filterbank ──────────────────────────────────────────
mel_fb = librosa.filters.mel(
    sr=sr, n_fft=n_fft, n_mels=80, fmin=0, fmax=8000
)  # shape: (80, 257)
S_mel = mel_fb @ S_power  # shape: (80, T)

# ── 5. Log compression ─────────────────────────────────────────
S_log_mel = np.log(S_mel + 1e-9)  # shape: (80, T)

# ── 6. MFCC (optional, classical) ─────────────────────────────
mfccs = librosa.feature.mfcc(S=S_log_mel, n_mfcc=13)
delta = librosa.feature.delta(mfccs)
delta2 = librosa.feature.delta(mfccs, order=2)
mfcc_full = np.vstack([mfccs, delta, delta2])  # shape: (39, T)

# ── 7. One-liner equivalents (librosa shorthand) ───────────────
S_log_mel_fast = librosa.power_to_db(
    librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_len, n_mels=80
    ),
    ref=np.max,
)

# ── 8. Visualize ───────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(10, 8))

librosa.display.waveshow(y, sr=sr, ax=axes[0])
axes[0].set_title("Waveform")

librosa.display.specshow(
    librosa.amplitude_to_db(np.abs(D)),
    sr=sr,
    hop_length=hop_len,
    x_axis="time",
    y_axis="hz",
    ax=axes[1],
)
axes[1].set_title("Spectrogram (linear freq)")

librosa.display.specshow(
    S_log_mel, sr=sr, hop_length=hop_len, x_axis="time", y_axis="mel", ax=axes[2]
)
axes[2].set_title("Log-Mel Spectrogram")

plt.tight_layout()
# plt.savefig("pipeline.png", dpi=150)